# Consistency gate — result explorer

Drill-down companion to `consistency_gate.py`. **The gate is the single source of
truth for executing cycles and computing scores** — this notebook only loads a gate
run's saved artifacts and rebuilds exploration dataframes from them, so there is one
metric implementation and the numbers here always match the gate's verdict.

Produce data first:

```bash
python consistency_gate.py --create-benchmark 5   # once: pin the benchmark set
python consistency_gate.py                        # run cycles + verdict
```

Then run this notebook to explore: which keys flapped, in which cycles, how values
varied, and which cycles were plan_degraded.


In [9]:
import json
from pathlib import Path

import pandas as pd

# Reuse the gate's flattening and scoring — never re-implement its metric here.
# Force-reload: the gate module evolves alongside this notebook, and a kernel
# that imported an older version would otherwise keep it cached.
import importlib
import consistency_gate as cg
cg = importlib.reload(cg)

RESULTS_ROOT = Path("data/results")

# Set explicitly to explore a specific run, else the latest gate run is used
# (falling back to legacy consistency_* notebook runs, rescored on the fly).
RUN_DIR: Path | None = None

if RUN_DIR is None:
    candidates = sorted(RESULTS_ROOT.glob("consistency_gate_*")) or sorted(
        RESULTS_ROOT.glob("consistency_*")
    )
    if not candidates:
        raise FileNotFoundError(
            "No consistency run under data/results — run consistency_gate.py first."
        )
    RUN_DIR = candidates[-1]
print(f"Exploring {RUN_DIR}")


Exploring data/results/consistency_20260712T084159Z


## Gate verdict

Read from `gate_result.json` when present.  Legacy `consistency_*` runs predate the
gate; for those the scores are rebuilt from the saved `task_*.json` files with the
gate's own `score_run_dir` — same metric, computed on the fly.  (Failed cycles save
no file, so rebuilt scores can't account for them.)


In [10]:
gate_file = RUN_DIR / "gate_result.json"
if gate_file.exists():
    gate = json.loads(gate_file.read_text())
else:
    gate = cg.score_run_dir(RUN_DIR)
    gate["verdict"] = "rescored from run files (no gate_result.json)"

print(f"verdict:              {gate['verdict']}")
print(f"avg key consistency:  {gate['avg_key_consistency_pct']}% "
      f"(threshold {gate.get('threshold_pct', 'n/a')}%)")
print(f"plan_degraded rate:   {gate['plan_degraded_pct']}% "
      f"(max {gate.get('max_degraded_pct', 'n/a')}%)")
print(f"kb_version:           {gate.get('kb_version', 'n/a')}")
print(f"cycles per task:      {gate.get('cycles', 'n/a')}   "
      f"benchmark: {gate.get('benchmark', 'n/a')}")


verdict:              rescored from run files (no gate_result.json)
avg key consistency:  88.9% (threshold n/a%)
plan_degraded rate:   6.7% (max n/a%)
kb_version:           5ec283c3eec3
cycles per task:      n/a   benchmark: n/a


In [11]:
# Per-key consistency table, straight from the gate's scoring output.
key_consistency = pd.DataFrame(
    [
        {"task_id": task_id, "task_name": task["task_name"], **key}
        for task_id, task in gate["tasks"].items()
        for key in task["keys"]
    ]
)
key_consistency.sort_values("consistency_pct") if not key_consistency.empty else key_consistency


,task_id,task_name,config_key,cycles_seen,cycles_ok,consistency_pct,distinct_values,value_spread_pct
0,1630937,aa-257 demographics,spark.sql.adaptive.coalescePartitions.minParti...,1,3,33.3,[128m],0.0
2,1630937,aa-257 demographics,spark.sql.shuffle.partitions,1,3,33.3,[4000],0.0
1,1630937,aa-257 demographics,spark.sql.adaptive.enabled,2,3,66.7,[true],NaN
3,2623786,stream,spark.driver.memory,3,3,100.0,"[10g, 12g, 28g]",180.0


## Per-cycle drill-down

Rebuilt from the per-run `task_*.json` files the gate saves (one per cycle).
Cycle numbers are reconstructed from `saved_at` order within each task.


In [12]:
def load_cycle_runs(run_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(run_dir.glob("task_*.json")):
        payload = json.loads(path.read_text())
        rows.append(
            {
                "task_id": payload["task_id"],
                "task_name": payload.get("task_name"),
                "saved_at": payload["saved_at"],
                "plan_degraded": bool(payload["plan"].get("plan_degraded")),
                "health_breach_unactioned": bool(payload["plan"].get("health_breach_unactioned")),
                "summary": payload["plan"].get("summary"),
                "n_unactioned": len(payload["plan"].get("unactioned_insights", [])),
                "recommendations": list(cg._iter_recommendations(payload["assembled_output"])),
                "file": path.name,
            }
        )
    runs = pd.DataFrame(rows)
    if runs.empty:
        return runs
    runs["cycle"] = runs.sort_values("saved_at").groupby("task_id").cumcount() + 1
    return runs.sort_values(["task_id", "cycle"]).reset_index(drop=True)

runs_df = load_cycle_runs(RUN_DIR)
runs_df[["task_id", "task_name", "cycle", "plan_degraded", "health_breach_unactioned", "n_unactioned", "summary"]]


,task_id,task_name,cycle,plan_degraded,health_breach_unactioned,n_unactioned,summary
0,1630937,aa-257 demographics,1,False,False,1,"2 recommendation(s), ordered health-first. Tie..."
1,1630937,aa-257 demographics,2,False,True,3,"1 recommendation(s), ordered health-first. Tie..."
2,1630937,aa-257 demographics,3,False,True,2,No recommendations passed safety checks.
3,1732445,compute,1,False,False,0,No insights detected.
4,1732445,compute,2,False,False,0,No insights detected.
5,1732445,compute,3,False,False,0,No insights detected.
6,2390906,landing_job,1,False,False,0,No insights detected.
7,2390906,landing_job,2,False,False,0,No insights detected.
8,2390906,landing_job,3,False,False,0,No insights detected.
9,2577699,landing_job,1,False,False,3,No recommendations passed safety checks.


In [13]:
# One row per recommended config_key per cycle.
recs_df = pd.DataFrame(
    [
        {"task_id": run.task_id, "task_name": run.task_name, "cycle": run.cycle, **rec}
        for run in runs_df.itertuples()
        for rec in run.recommendations
    ]
)
recs_df.head(20)


,task_id,task_name,cycle,config_key,suggested_value,blocked
0,1630937,aa-257 demographics,1,spark.sql.adaptive.enabled,true,False
1,1630937,aa-257 demographics,1,spark.sql.adaptive.coalescePartitions.minParti...,128m,False
2,1630937,aa-257 demographics,2,spark.sql.adaptive.enabled,true,False
3,1630937,aa-257 demographics,3,spark.sql.shuffle.partitions,4000,True
4,2623786,stream,1,spark.driver.memory,28g,False
5,2623786,stream,2,spark.driver.memory,10g,False
6,2623786,stream,3,spark.driver.memory,12g,False


In [14]:
# Presence matrix: which cycle recommended which key — flapping keys show as gaps.
if recs_df.empty:
    print("No recommendations in this run.")
else:
    presence = (
        recs_df.assign(present=1)
        .pivot_table(index=["task_id", "config_key"], columns="cycle", values="present", fill_value=0)
        .astype(int)
    )
    display(presence)


cycle                                                       1  2  3
task_id config_key                                                 
1630937 spark.sql.adaptive.coalescePartitions.minPartit...  1  0  0
        spark.sql.adaptive.enabled                          1  1  0
        spark.sql.shuffle.partitions                        0  0  1
2623786 spark.driver.memory                                 1  1  1

In [15]:
# Value drill-down: suggested_value per key per cycle, for keys seen in >= 2 cycles.
if not recs_df.empty:
    repeated = recs_df.groupby(["task_id", "config_key"])["cycle"].transform("nunique") >= 2
    values = (
        recs_df[repeated]
        .pivot_table(
            index=["task_id", "config_key"], columns="cycle",
            values="suggested_value", aggfunc="first",
        )
    )
    display(values)


,cycle,1,2,3
task_id,config_key,,,
1630937,spark.sql.adaptive.enabled,true,true,NaN
2623786,spark.driver.memory,28g,10g,12g


## Ad-hoc: measure fresh tasks

To explore tasks outside the pinned benchmark, sample a throwaway benchmark and run
the gate's own loop — same execution path, same corrected metric — then re-run this
notebook on the new output directory.


In [16]:
# import asyncio
# from datetime import datetime, timezone
# import consistency_gate as cg
#
# adhoc_benchmark = Path("data/benchmark_adhoc.json")
# cg.create_benchmark(adhoc_benchmark, n_tasks=3)
# jobs = cg.load_benchmark(adhoc_benchmark)
# stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
# out = RESULTS_ROOT / f"consistency_gate_adhoc_{stamp}"
# out.mkdir(parents=True)
# records, kb_version = await cg.run_cycles(jobs, cycles=3, output_dir=out)
# result = cg.score_all(records, jobs)
# result.update(kb_version=kb_version, cycles=3, threshold_pct=None,
#               max_degraded_pct=None, benchmark=str(adhoc_benchmark),
#               run_utc=stamp, verdict="adhoc (not gated)")
# (out / "gate_result.json").write_text(json.dumps(result, indent=2))
# print(f"explore with RUN_DIR = Path('{out}')")
